# Step 3 – Live Line Following

Loads the trained ResNet18 model and runs it on live camera frames to steer the robot.
Depth data from the ZED camera provides a **collision safety stop**.

**Prerequisites:** `line_follower.pth` must exist (produced by Step 2).

**System overview:**
```
ZED Camera → colour frame → ResNet18 → steering decision (left / forward / right)
```

In [ ]:
# ── Cell 1: Load model ────────────────────────────────────────────────────────
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn

SAVE_PATH = 'line_follower.pth'
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ckpt        = torch.load(SAVE_PATH, map_location=device, weights_only=True)
class_names = ckpt['class_names']   # ['forward', 'left', 'right'] — alphabetical
print('Classes:', class_names)
print(f'Checkpoint val_acc: {ckpt["val_acc"]:.3f}')

model = torchvision.models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(class_names))
model.load_state_dict(ckpt['model_state_dict'])
model = model.to(device)
model.eval()

# Same normalisation used during training
infer_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])
print('Model ready.')

In [ ]:
# ── Cell 2: Start camera ──────────────────────────────────────────────────────
import cv2, threading, time, numpy as np
import pyzed.sl as sl
import traitlets
from traitlets.config.configurable import SingletonConfigurable

class Camera(SingletonConfigurable):
    color_value = traitlets.Any()

    def __init__(self):
        super().__init__()
        self.zed = sl.Camera()
        init = sl.InitParameters()
        init.camera_resolution = sl.RESOLUTION.VGA
        init.depth_mode = sl.DEPTH_MODE.NONE
        init.coordinate_units = sl.UNIT.MILLIMETER
        status = self.zed.open(init)
        if status != sl.ERROR_CODE.SUCCESS:
            print('Camera open failed:', status)
            exit(1)
        self.runtime = sl.RuntimeParameters()
        self.thread_runnning_flag = False
        info = self.zed.get_camera_information()
        self.width  = info.camera_configuration.resolution.width
        self.height = info.camera_configuration.resolution.height
        self.image  = sl.Mat(self.width, self.height, sl.MAT_TYPE.U8_C4, sl.MEM.CPU)

    def _capture_frames(self):
        while self.thread_runnning_flag:
            if self.zed.grab(self.runtime) == sl.ERROR_CODE.SUCCESS:
                self.zed.retrieve_image(self.image, sl.VIEW.LEFT)
                raw = self.image.get_data()
                self.color_value = cv2.cvtColor(raw, cv2.COLOR_BGRA2BGR)

    def start(self):
        if not self.thread_runnning_flag:
            self.thread_runnning_flag = True
            self.thread = threading.Thread(target=self._capture_frames)
            self.thread.start()

    def stop(self):
        if self.thread_runnning_flag:
            self.thread_runnning_flag = False
            self.thread.join()

def bgr8_to_jpeg(img):
    return bytes(cv2.imencode('.jpg', img)[1])

# Clear any leftover singleton from a previous run
if Camera._instance is not None:
    try:
        Camera._instance.stop()
        Camera._instance.zed.close()
    except Exception:
        pass
    Camera.clear_instance()

camera = Camera()
camera.start()
time.sleep(1)  # let camera warm up
print('Camera started.')

In [ ]:
# ── Cell 3: Live line-following loop ──────────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display
import motors

robot = motors.MotorsYukon(mecanum=False)

# ── Tunable parameters ────────────────────────────────────────────────────────
FORWARD_SPEED  = 0.40    # straight-line speed (0–1)
TURN_SPEED     = 0.35    # turning speed
CONFIDENCE_THR = 0.40    # lowered — acts on weaker predictions rather than stopping
CROP_START     = 3       # must match Step 1 — crop from 1/3 down

# Display widgets
display_color = widgets.Image(format='jpeg', width='50%')
status_label  = widgets.Label(value='Starting...')
display(widgets.VBox([display_color, status_label]))

# ── Helper: run one CNN inference step ────────────────────────────────────────
def predict_steering(frame):
    """Return (class_name, confidence) for the current frame."""
    h = frame.shape[0]
    crop = frame[h//CROP_START:, :]
    rgb  = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    x    = infer_tf(rgb).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x)
    probs = torch.softmax(logits, dim=1)[0]
    idx   = probs.argmax().item()
    return class_names[idx], probs[idx].item()

# ── Main inference loop ───────────────────────────────────────────────────────
print('Running line follower. Interrupt kernel to stop.')
try:
    while True:
        if camera.color_value is None:
            time.sleep(0.05)
            continue

        frame = camera.color_value.copy()
        steering, conf = predict_steering(frame)

        if conf < CONFIDENCE_THR:
            robot.stop()
            action = f'STOP (low conf {conf:.2f})'
        elif steering == 'forward':
            robot.forward(FORWARD_SPEED)
            action = 'FORWARD'
        elif steering == 'left':
            robot.left(TURN_SPEED)
            action = 'LEFT'
        elif steering == 'right':
            robot.right(TURN_SPEED)
            action = 'RIGHT'
        else:
            robot.stop()
            action = 'STOP'

        disp = cv2.resize(frame, None, fx=0.3, fy=0.3)
        cv2.putText(disp, f'{action}  {conf:.2f}',
                    (5, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
        display_color.value = bgr8_to_jpeg(disp)
        status_label.value  = f'Action: {action}  Conf: {conf:.2f}'

except KeyboardInterrupt:
    print('Interrupted by user.')
finally:
    robot.stop()
    print('Robot stopped.')

In [ ]:
# ── Cell 4: Stop camera (always run this when done) ───────────────────────────
robot.stop()
camera.stop()
print('Camera and robot stopped.')